In [1]:
import os
import sys
import json

sys.path.append(os.path.abspath(".."))
os.environ['CHROMA_DB_PATH'] = os.path.join(os.path.abspath('..'), "chroma_db")

from model.rag_4b import solve_math_direct_4b, solve_math_question_4b

d:\User\ProjectGithub\hiepnguyenn-99\RAG-Solve-Math\appenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open('../data/cleaned/test/dai_so.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
test_data_math = data["problems"]
print(f"Số lượng bài test: {len(test_data_math)}")

Số lượng bài test: 354


In [3]:
def solve_math_batch(questions):
    """
    Nhận list câu hỏi, trả về list đáp án bằng cách gọi solve_math_question_4b từng câu.
    Nếu gặp lỗi, trả về None cho câu đó.
    """
    answers = []
    for q in questions:
        try:
            result = solve_math_direct_4b(q)
            ans = result if isinstance(result, str) else result[0]
        except Exception as e:
            ans = None
        answers.append(ans)
    return answers

In [4]:
def process_in_batches(questions, batch_size):
    """
    """
    all_answers = []
    for i in range(0, len(questions), batch_size):
        batch = questions[i:i+batch_size]
        batch_answers = solve_math_batch(batch)
        all_answers.extend(batch_answers)
    return all_answers

TEST giải toán

In [5]:
# Test giải toán trên dữ liệu test và ghi kết quả từng câu ngay sau khi xử lý
# batch_size = 32
batch_size = 1
questions = [item['question'] for item in test_data_math[-100:]]
start = 65
end = len(questions)
os.makedirs('Answer', exist_ok=True)
with open('Answer/qwen4b8m-ft/dai_so.json', 'a', encoding='utf-8') as f:
    if start == 0:
        f.write('[\n')
    for idx in range(start, end, batch_size):
        batch = questions[idx:idx+batch_size]
        answers = process_in_batches(batch, batch_size=len(batch))
        for i, (question, answer) in enumerate(zip(batch, answers), idx+1):
            obj = {"index": i, "input": question, "output": answer}
            if i == end:
                f.write(json.dumps(obj, ensure_ascii=False) + '\n')
            else:
                f.write(json.dumps(obj, ensure_ascii=False) + ',\n')
        print(f"Đã ghi gần đến câu thứ {idx+len(batch)}")
    f.write(']\n')

Đã ghi gần đến câu thứ 66
Đã ghi gần đến câu thứ 67
Đã ghi gần đến câu thứ 68
Đã ghi gần đến câu thứ 69
Đã ghi gần đến câu thứ 70
Đã ghi gần đến câu thứ 71
Đã ghi gần đến câu thứ 72
Đã ghi gần đến câu thứ 73
Đã ghi gần đến câu thứ 74
Đã ghi gần đến câu thứ 75
Đã ghi gần đến câu thứ 76
Đã ghi gần đến câu thứ 77
Đã ghi gần đến câu thứ 78
Đã ghi gần đến câu thứ 79
Đã ghi gần đến câu thứ 80
Đã ghi gần đến câu thứ 81
Đã ghi gần đến câu thứ 82
Đã ghi gần đến câu thứ 83
Đã ghi gần đến câu thứ 84
Đã ghi gần đến câu thứ 85
Đã ghi gần đến câu thứ 86
Đã ghi gần đến câu thứ 87
Đã ghi gần đến câu thứ 88
Đã ghi gần đến câu thứ 89
Đã ghi gần đến câu thứ 90
Đã ghi gần đến câu thứ 91
Đã ghi gần đến câu thứ 92
Đã ghi gần đến câu thứ 93
Đã ghi gần đến câu thứ 94
Đã ghi gần đến câu thứ 95
Đã ghi gần đến câu thứ 96
Đã ghi gần đến câu thứ 97
Đã ghi gần đến câu thứ 98
Đã ghi gần đến câu thứ 99
Đã ghi gần đến câu thứ 100


In [6]:
# Kiểm tra lại file answer RAG đã ghi ra
with open('Answer/qwen4b8m-ft/dai_so.json', 'r', encoding='utf-8') as f:
    answer_rag_data = json.load(f)
print(f"Số lượng kết quả trong file Answer/qwen4b8m-ft/dai_so.json RAG: {len(answer_rag_data)}")

Số lượng kết quả trong file Answer/qwen4b8m-ft/dai_so.json RAG: 100
